# Chain-of-Thought and Thinking Models

> What is 357 × 289? Even a human has to line up the long multiplication step by step. A standard next-token LLM, without an explicit reasoning process, often has to predict the final answer directly. For multi-step math, logic, and code problems, this compresses a lot of intermediate computation into a single response, which is error-prone.
>
> Chain-of-Thought (CoT) has the model write out its intermediate steps before giving the answer, which boosts accuracy substantially. Thinking models like o1 and DeepSeek-R1 go further: they do not just add thinking at inference time; they learn to "think before answering" during training. This is a training-side change. The other path is to leave the model untouched and only spend more compute at inference time — Best-of-N, search, and verifiers all belong to this category. This section breaks down the principles behind both routes.

The core idea of CoT can be stated in one sentence: demonstrate or instruct in the prompt to "write the intermediate steps first, then give the final answer," so that intermediate results also enter the context. The classic source is [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903).

Writing out intermediate steps helps because each step shrinks the problem, and if some intermediate step goes wrong, later steps still have a chance to correct it. Classic CoT prompting does not change the model parameters; but if the model has seen a large amount of reasoning-style data during pretraining, SFT, or RL, it may also have already learned to expand steps when facing hard problems.

Thinking models go further — during training they are shaped by reinforcement learning into a behavior pattern of "first produce an internal chain of thought, then output the final answer." The key insight from DeepSeek-R1 is that on verifiable tasks like math and code, a suitable RL reward can reinforce longer, more self-checking reasoning trajectories; but this depends on the base model, the reward design, the sampling scale, and training stability. Reference: [DeepSeek-R1](https://arxiv.org/abs/2501.12948).


## 1. The Reasoning Gap in LLMs

```
User: 357 × 289 = ?

Inside a standard LLM:
  input tokens → embedding → transformer × N → output "103173"

  Calling this a "guess" is a teaching metaphor. The model is indeed doing next-token prediction; it may also have learned generalizable computation patterns. The point is: without generating intermediate steps, complex computation is more easily compressed into a single direct prediction.
  If the specific combination 357×289 never appeared in the training data, it will usually guess wrong.
```

**Root cause**: Every layer of a Transformer does a fixed amount of computation. Whether the question is "1+1" or "calculus," a single forward pass has a fixed number of layers and limited computational depth. When generating CoT, the model can feed the intermediate results it just produced back into the context — effectively trading more output tokens for more reasoning steps.

Just like asking you to mentally compute 357×289, you cannot give the answer directly. You need:
1. 357×200 = 71400
2. 357×80 = 28560
3. 357×9 = 3213
4. 71400 + 28560 + 3213 = 103173

An LLM also needs this "step-by-step computation" process — that is exactly what CoT provides.


## 2. Chain-of-Thought (CoT)

The core idea of CoT is extremely simple: **demonstrate "write the reasoning process first, then give the answer" in the prompt**.

```
Prompt without CoT:
  Q: 357 × 289 = ?
  A: 103173  ← model guesses directly

Prompt with CoT (Few-shot):
  Q: 123 × 45 = ?
  A: 123×40=4920, 123×5=615, 4920+615=5535. The answer is 5535.

  Q: 357 × 289 = ?
  A:  ← the model imitates the format above, writing the process first, then the answer
```

**Why does it work?** Because when the model generates the reasoning process, the result of each intermediate step becomes "context" for the subsequent steps. The Transformer's attention can see the intermediate results computed earlier and continue reasoning based on them.

In essence: **CoT turns "a problem one forward pass cannot solve" into "multiple forward passes working in relay."**


In [ ]:
# Live computation demo: why CoT works
import numpy as np
import random

print("=== Why CoT Works ===")
print()

target = 357 * 289

# Without CoT: model must compute in one step
print("Without CoT:")
print(f"  Model must compute 357 × 289 in one step")
print(f"  Correct answer: {target:,}")
np.random.seed(42)
guesses = np.random.randint(80000, 120000, size=5)
closest = guesses[np.argmin(np.abs(guesses - target))]
print(f"  Model can only 'guess': {closest:,} (off by {abs(closest - target):,})")
print(f"  → One shot is too hard!")
print()

# With CoT: step-by-step
print("With CoT (step by step):")
result = 0
steps = []
for multiplier, label in [(200, "200"), (80, "80"), (9, "9")]:
    partial = 357 * multiplier
    result += partial
    steps.append(partial)
    print(f"  357 × {label} = {partial:,}")

total = sum(steps)
print(f"  {' + '.join(f'{s:,}' for s in steps)} = {total:,}")
print()
print(f"✅ CoT result: {total:,} == Correct answer {target:,}")
print()
print("Key insight: each step's result can be referenced by subsequent steps.")
print("This trades 'token generation time' for 'computational depth.'")


### 2.5 Self-Consistency: Ask the Same Question Multiple Times, Vote on the Answer

CoT has one problem: **the model might make an error on a particular reasoning chain**.
Ask the model the same math question 5 times (with temperature > 0), and it may produce 3 different reasoning processes and answers.

**The core idea of Self-Consistency**:
```
Same question → sample N different CoT reasoning chains → vote on the final answer → the answer with the most votes wins
```

**Why does it work?**

Intuition: you take a hard problem and ask 5 classmates —
- Each classmate might go wrong at some step
- But different classmates are **likely to make different errors, unlikely to make the same error**
- The correct answer is unique, and it appears most frequently across multiple correct or partially correct chains

Mathematically: if a single chain's accuracy is p, the probability that the majority of N chains are correct increases with N:
```
p=0.6, N=1: accuracy 60%
p=0.6, N=5: probability of at least 3 correct = C(5,3)p^3(1-p)^2 + ... ≈ 68%
p=0.7, N=5: probability of at least 3 correct ≈ 84%  ← significant improvement!
```

**What you do NOT need**: no retraining, no architecture changes — just **multiple sampling + voting** at inference time.

The following simulation demonstrates the full process:


In [ ]:
# ============================================================
# Self-Consistency demo: same question, 5 reasoning chains, vote on the answer
# ============================================================
import random
random.seed(42)

print("=" * 70)
print("Problem: Xiaoming has 15 apples. He gives Xiaohong 3, buys 8 more,")
print("         then eats 2, and finally gives half of the remainder to Xiaogang.")
print("         How many apples does Xiaoming have now?")
print("=" * 70)

# Simulate 5 different reasoning chains (mimicking model sampling at temperature > 0)
# Each chain shows "reasoning process" + "final answer"

reasoning_chains = [
    {
        "id": 1,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 2,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 3,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → remainder 18 - 9 = 9",  # reasoning correct, answer correct
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 4,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: forgot to divide by 2! → answer 18",  # ← forgot the last step!
        ],
        "answer": 18,
        "correct": False
    },
    {
        "id": 5,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3, buys 8 → total 15 - 3 + 8 = 20",
            "Step 3: eats 2 → 20 - 2 = 18",
            "Step 4: gives half to Xiaogang → gives 18/2 = 9, keeps 9",
        ],
        "answer": 9,
        "correct": True
    },
]

# Print each reasoning chain
for chain in reasoning_chains:
    print(f"\n--- Chain #{chain['id']} ---")
    for step in chain['reasoning']:
        print(f"  {step}")
    status = "✓ correct" if chain['correct'] else "✗ wrong"
    print(f"  Answer: {chain['answer']} apples  {status}")

# ============================================================
# Majority voting
# ============================================================
print(f"\n{'=' * 70}")
print("Voting")
print(f"{'=' * 70}")

from collections import Counter
answers = [c['answer'] for c in reasoning_chains]
vote_counts = Counter(answers)

for ans, count in vote_counts.most_common():
    correct_mark = "✓" if ans == 9 else "✗"
    bar = "█" * count
    print(f"  Answer {ans}: {count} votes {bar} {correct_mark}")

winner = vote_counts.most_common(1)[0][0]
winner_is_correct = (winner == 9)

print(f"\n  🏆 Final answer (majority vote): {winner} apples")
print(f"     Correct: {'✓ correct!' if winner_is_correct else '✗ wrong'}")

# ============================================================
# Comparison: single vs Self-Consistency
# ============================================================
print(f"\n{'=' * 70}")
print("Comparison")
print(f"{'=' * 70}")
print(f"  Single sampling (pick one at random): accuracy = 4/5 = 80%")
print(f"  Self-Consistency (5 chains vote): accuracy = 100% (all correct this time!)")
print(f"")
print(f"  Key insight:")
print(f"  - Chain #4 made an error at the last step (forgot to divide by 2)")
print(f"  - But the other 4 chains were correct, so the vote result = 9 (correct)")
print(f"  - Self-Consistency swallowed that error!")
print(f"")
print(f"  This is the power of Self-Consistency:")
print(f"  'Majority rules' — one chain making an error is fine; multiple chains will not make the same mistake.")


#### 2.5.1 Self-Consistency vs Thinking Models

Self-Consistency and the Thinking models (o1, R1) discussed later are fundamentally different:

| | Self-Consistency | Thinking Models |
|------|---------|-------|
| **How** | Multiple sampling + voting at inference time | Learns to "self-reflect" during training |
| **Cost** | N× inference cost (sample N times) | 1 inference, but internal thinking is long |
| **Training needed** | No | Yes (RL training) |
| **Key capability** | Diverse sampling | Self-verification, error correction, backtracking |

**In essence**:
- Self-Consistency is **external** error correction — relying on multiple samples to "hit" the correct answer
- Thinking models are **internal** error correction — checking and correcting within a single reasoning chain

The latter is harder to train, but more efficient at inference (no repeated sampling needed).
The self-verification of R1-style models is better understood as "checking and backtracking within a single long reasoning chain." It and Self-Consistency both exploit multi-step / multi-sample information, but the mechanisms differ and they cannot simply be equated. Self-Consistency reference: [Wang et al. 2022](https://arxiv.org/abs/2203.11171).


## 3. From CoT to Thinking Models

The problem with CoT is that the thinking process is exposed to the user. Users do not necessarily want to see your scratch paper.

The approach of thinking models is to separate the "scratch paper" from the "final answer":

```text
What the user sees:
  Q: 357 × 289 = ?
  A: 103173

What the model actually generates internally:
  <think>
  357×200=71400
  357×80=28560
  357×9=3213
  71400+28560+3213=103173
  </think>
  103173
```

The `<think>` and `</think>` here are thinking markers.
They act like a pair of brackets: the thinking draft goes in between, and the final answer follows.

The frontend or API can choose to display only the answer after `</think>`, collapsing or hiding the intermediate draft.

**In essence**: Thinking models = engineered packaging of CoT. The thinking process is still there, just partitioned and managed by special markers.


### 3.5 How Think Markers Work Under the Hood

#### 3.5.1 Why `<think>` is not ordinary text

You might think: isn't `<think>` just a few characters? The model just outputs these characters and that's it?

It is not that simple.

If `<think>` were just ordinary text, three problems could arise:

1. **Easily fragmented by the tokenizer**: `<think>` might become three tokens: `<`, `think`, `>`.
2. **Unstable end boundary**: The model might accidentally write `</think>` inside the draft, causing the frontend to end thinking prematurely.
3. **Hard to control during training**: It is difficult to stably distinguish between the thinking zone and the answer zone.

So the more robust approach is to add `<think>` and `</think>` as **special tokens**. But this is not the only option; some models also use plain strings, chat templates, API blocks, or separate fields to distinguish reasoning from answer.

That is, they should have their own independent IDs, just like `<BOS>` and `<EOS>`:

```text
<think>   -> 100
</think>  -> 101
```

This way, when the model generates ID 100, it enters the thinking zone; when it generates ID 101, it exits the thinking zone.


#### 3.5.2 Current Practice: How to Add New Thinking Markers to Training

In modern training, adding new markers like `<think>` generally falls into two scenarios.

**Scenario A: Training tokenizer and model from scratch**

Add these markers to the tokenizer's special token list from the start:

```text
<BOS>, <EOS>, <PAD>, <think>, </think>
```

This way, during tokenizer training they will not be fragmented; during model training, they will also learn embeddings like regular tokens.

**Scenario B: Continuing training on an existing model**

This is the more common path. For example, if you take an open-source base model and want it to learn a new thinking format, you typically do 4 things:

1. **Add markers to the tokenizer**: Add `<think>` and `</think>` as special tokens.
2. **Expand model embeddings**: The tokenizer vocabulary grows, so the model's `Embedding` and output layers must also grow.
3. **Prepare formatted data**: Training samples must actually contain `<think>...</think>`.
4. **Continue SFT / RL**: Let the model learn via loss or reward when to open and when to close thinking.

Common pseudocode in practice:

```python
new_tokens = {"additional_special_tokens": ["<think>", "</think>"]}
num_added = tokenizer.add_special_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))

# Then continue training with data containing <think>...</think>
```

One thing that is easy to misunderstand here: **adding tokens just gives the model a new pen; it does not mean it can write reasoning.**
What actually teaches the model to think is the training data, loss mask, and reward design that follow.


In [ ]:
# Minimal example: training sample after adding <think> markers
vocab = {
    "<BOS>": 0,
    "<EOS>": 1,
    "<PAD>": 2,
    "user": 3,
    "assistant": 4,
    "answer": 5,
    "357": 6,
    "289": 7,
    "103173": 8,
}

new_symbols = ["<think>", "</think>"]
for symbol in new_symbols:
    if symbol not in vocab:
        vocab[symbol] = len(vocab)

train_tokens = [
    "<BOS>",
    "user", "357", "289",
    "assistant", "<think>", "357", "289", "103173", "</think>",
    "answer", "103173",
    "<EOS>",
]
train_ids = [vocab[token] for token in train_tokens]

print("New symbols:")
for symbol in new_symbols:
    print(f"  {symbol} -> ID {vocab[symbol]}")

print()
print("Training sample:")
print(train_tokens)
print()
print("Training IDs:")
print(train_ids)
print()
print("Key observation: the model does not understand the English word '<think>',")
print("but learns through training that between ID 9 and ID 10 it should write a reasoning draft.")


In [ ]:
# ============================================================
# Demo: How Special Tokens are tokenized
# ============================================================
print("=== Special Token vs Plain Text: Tokenization Comparison ===\n")

# Simulate tokenizer behavior -- simplified BPE-style tokenizer
# Core question: what happens if <think> is just plain text?

# Assume this is the tokenizer vocabulary (simplified)
vocab = {
    "I": 1, "like": 2, "eat": 3, "apple": 4, "orange": 5,
    "think": 7, "<": 9, ">": 10, "/": 11,
    "reasoning": 13, "process": 14, "answer": 15, "is": 16,
    # ↓ Key: if set as a special token, it gets a unique ID
    # "<think>": 100,
    # "</think>": 101,
}

print("[Vocabulary] Plain tokens (no special tokens):")
for k, v in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  '{k}' → {v}")
print()

# ========================================
# Scenario 1: without special tokens
# ========================================
print("=" * 55)
print("Scenario 1: <think> is just plain text (not added to vocabulary)")
print("=" * 55)

def tokenize_plain(text, vocab):
    """Simulate a BPE/word-level tokenizer without special tokens"""
    tokens = []
    i = 0
    while i < len(text):
        # Longest match for regular tokens
        matched = None
        for word in sorted(vocab.keys(), key=len, reverse=True):
            if text[i:].startswith(word):
                matched = (word, vocab[word])
                break
        if matched:
            tokens.append(matched)
            i += len(matched[0])
        else:
            # Character-level fallback
            ch = text[i]
            tid = ord(ch) % 50 + 20  # Simulate unknown token id
            tokens.append((ch if ch != ' ' else '⎵', tid))
            i += 1
    return tokens

text = "Ilike<think>reasoningprocess</think>answerisapple"
tokens_plain = tokenize_plain(text, vocab)

print(f"Input: {text}")
print(f"\nTokenization result ({len(tokens_plain)} tokens):")
ids = []
for word, tid in tokens_plain:
    ids.append(tid)
    flag = "⚠️ fragmented" if tid in [9,10,11] else ""
    print(f"  [{word:6s}] → ID {tid:3d} {flag}")

print(f"\nToken ID sequence: {ids}")
print(f"\nKey problems:")
print(f"  ❌ '<think>' is split into 3 tokens: < think >   (3 IDs in total)")
print(f"  ❌ If '<' appears inside thinking content (e.g. x<5) → chaos")
print(f"  ❌ During training, cannot use token IDs to precisely locate 'where thinking starts'")
print(f"  ❌ The model must learn to 'output exactly these 5 tokens in order' → extremely hard to learn")

# ========================================
# Scenario 2: with special tokens added
# ========================================
print(f"\n{'='*55}")
print("Scenario 2: <think> set as Special Token (one of the more robust approaches)")
print("=" * 55)

# Add special tokens to vocabulary
vocab_with_special = vocab.copy()
vocab_with_special["<think>"] = 100
vocab_with_special["</think>"] = 101

print("Added to vocabulary:")
print("  '<think>'  → 100  ★ special token")
print("  '</think>' → 101  ★ special token")
print()

def tokenize_with_special(text, vocab):
    """
    The tokenizer scans for special tokens first (longest match priority),
    then scans for regular tokens. This is how all real tokenizers work.
    """
    tokens = []
    i = 0
    while i < len(text):
        # Step 1: match special tokens first (ID >= 100)
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: (-len(x), x)):
            if text[i:].startswith(word):
                matched = (word, vocab[word])
                break
        if matched:
            tokens.append(matched)
            i += len(matched[0])
        else:
            ch = text[i]
            tokens.append((ch, ord(ch) % 50 + 20))
            i += 1
    return tokens

tokens_special = tokenize_with_special(text, vocab_with_special)

print(f"Input: {text}")
print(f"\nTokenization result ({len(tokens_special)} tokens):")
ids_special = []
for word, tid in tokens_special:
    ids_special.append(tid)
    flag = "★ special" if tid >= 100 else ""
    print(f"  [{word:12s}] → ID {tid:3d} {flag}")

print(f"\nToken ID sequence: {ids_special}")
print(f"Token count: {len(tokens_plain)} → {len(tokens_special)}")
print(f"\nKey advantages:")
print(f"  ✅ '<think>' = one token (ID=100), not fragmented")
print(f"  ✅ '<' or '>' inside thinking content have no effect (special matched first)")
print(f"  ✅ Check ID==100/101 to precisely locate thinking boundaries")
print(f"  ✅ Model only needs to learn to 'output token 100 at the right position'")

# ========================================
# Demo with real tokenizer (if tiktoken is available)
# ========================================
print(f"\n{'='*55}")
print("Real tokenizer approach")
print("=" * 55)

try:
    import tiktoken

    # GPT-2 tokenizer (no think special token)
    enc = tiktoken.get_encoding("gpt2")
    plain_result = enc.encode("<think>")
    print(f"\nGPT-2 tokenizer encoding '<think>':")
    print(f"  Token IDs: {plain_result}")
    print(f"  Each token: {[enc.decode([t]) for t in plain_result]}")
    print(f"  → Split into {len(plain_result)} tokens (because '<think>' is not in the GPT-2 vocabulary)")

    # Explain the real approach
    print(f"\nDeepSeek-R1 / Qwen3 approach:")
    print(f"  1. Add special tokens to the tokenizer vocabulary:")
    print(f"     tokenizer.add_special_tokens({{'<think>': ..., '</think>': ...}})")
    print(f"  2. Expand model embedding matrix (add a row for the new token)")
    print(f"  3. Now '<think>' is a single complete token")

except ImportError:
    print("\n(tiktoken not installed, skipping real tokenizer demo)")
    print("pip install tiktoken to run the demo above")
    print()
    print("Using GPT-4's cl100k_base as an example:")
    print("  '<think>' if not in vocabulary → may be split into multiple BPE tokens")
    print("  After adding to vocabulary → 1 token → model learning cost reduced 5x")

print(f"\nSummary: add_special_tokens() is the first step in training a thinking model.")


### 3.6 How to Compute Loss When Training Thinking Models

With special tokens like `<think>`, an important question arises:

**During training, should the tokens in the thinking section contribute to the loss?**

There are three strategies, each with tradeoffs. Let us first look at a concrete example:

```
Complete assistant output token sequence:
[<think>, 3, 1, 2, ×, 2, 0, 0, =, 7, 1, 4, 0, 0, </think>, 1, 0, 3, 1, 7, 3]
│←──────── thinking tokens ────────────→│←─ answer tokens ─→│
        14 tokens                             6 tokens
```

#### Strategy comparison

```
┌─────────────────────┬──────────────────┬──────────────────┬──────────────────┐
│                     │ Strategy A:      │ Strategy B:      │ Strategy C:      │
│                     │ Full Loss        │ Answer Only      │ Selective        │
│                     │ (all in loss)    │ (answer only)    │ Weighting        │
├─────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ thinking token      │       ✅ yes     │     ❌ no        │   🔶 yes but     │
│ loss                │                  │                  │   down-weighted  │
│                     │                  │                  │   (×0.1)         │
│ answer token        │       ✅ yes     │      ✅ yes      │     ✅ yes       │
│ loss                │                  │                  │                  │
│ special token       │       ✅ yes     │     ❌ no        │    ❌ no         │
│ loss (<think> etc.)│                  │                  │                  │
├─────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Advantage           │ Simple and direct│ Only cares about │ Balances the two │
│                     │ Thinking quality │ the final answer;│ Thinking is      │
│                     │ is also optimized│ thinking has high│ guided but does  │
│                     │                  │ freedom          │ not dominate     │
├─────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Disadvantage        │ Thinking may     │ Thinking process │ Requires tuning  │
│                     │ turn into        │ may degrade or   │ a weight         │
│                     │ "performance"    │ become incoherent│ hyperparameter   │
│                     │ rather than      │                  │                  │
│                     │ genuine reasoning│                  │                  │
├─────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Representative      │ DeepSeek-R1      │ Early thinking   │ Some RL          │
│ model / scenario    │ (RL phase, all   │ experiments      │ experiments      │
│                     │ in loss)         │                  │                  │
└─────────────────────┴──────────────────┴──────────────────┴──────────────────┘
```

#### SFT and RL phases must be understood separately

In the SFT phase, full loss / answer-only / selective weighting are all up for discussion: if the thinking zone is not supervised at all, the model may only learn the final answer format and the thinking behavior will be unstable.

In the RL phase, it is not ordinary CE loss "counting all tokens." The model first samples a response, then optimizes the policy based on the result reward; whether longer reasoning gets reinforced depends on the reward, format constraints, sampling length, and training stability.

But full loss also has a cost: the model may learn "performative thinking" — generating a pile of tokens that look like reasoning but are actually useless, simply because those tokens have low loss (the model is confident about them).
RL's outcome reward can suppress some useless reasoning, but it cannot guarantee that all thinking is genuinely effective; format checks, length control, and task evaluation are still needed.

#### How to implement it?

The core is constructing a **loss_mask** — a 0/1 array with the same shape as labels:

```python
# labels:   [100, 3, 1, 2, 11, ..., 101, 1, 0, 3, 1, 7, 3]
# loss_mask:[ 0,  1, 1, 1, 1,  ...,  0, 1, 1, 1, 1, 1, 1]
#             ↑ special token excluded          ↑ special token excluded
#
# final loss = cross_entropy(logits, labels) * loss_mask
# → special token positions have loss=0 → no gradient
```

Below is a complete code implementation of all three strategies.


In [ ]:
# ============================================================
# Complete implementation: loss_mask construction for three strategies
# ============================================================
import torch
import torch.nn.functional as F

import torch.nn as nn

print("=== Thinking Model Loss Mask Complete Implementation ===\n")

# Simulate a batch of token sequences
# Special token IDs: <think>=100, </think>=101
# Regular tokens: 1-50
# PAD=0, IGNORE=-100 (PyTorch standard ignore value)
THINK_START = 100
THINK_END = 101
IGNORE = -100

# Simulated training data: labels for two samples
# Sample 1: <think> 3,1,2,11,2,0,0 </think> 1,0,3,1,7,3  ← normal thinking→answer
# Sample 2: <think> 5,×,6,=,3,0 </think> 3,0              ← brief thinking→answer
batch_labels = torch.tensor([
    [100,   3,   1,   2,  11,   2,   0,   0, 101,   1,   0,   3,   1,   7,   3,   0],
    [100,   5,  12,   6,  13,   3,   0, 101,   3,   0,   0,   0,   0,   0,   0,   0],
])  # shape: [2, 16]

# Simulated model output logits
VOCAB_SIZE = 200
logits = torch.randn(2, 16, VOCAB_SIZE)

print("Batch labels:")
print(batch_labels)
print()

# ============================================================
# Strategy A: Full Loss — all tokens contribute to loss
# ============================================================
print("=" * 60)
print("Strategy A: Full Loss (all count)")
print("=" * 60)

def make_full_loss_mask(labels, ignore_id=0):
    """Simplest mask: only ignore PAD tokens"""
    return (labels != ignore_id).float()

mask_full = make_full_loss_mask(batch_labels)
print("loss_mask (0=PAD excluded, 1=included):")
print(mask_full.int())
print()

# Compute loss
loss_full = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    batch_labels.view(-1),
    ignore_index=0,  # PAD excluded
    reduction='none'
).view(2, 16)

print("Per-position loss:")
print(loss_full)
print()

total_full = loss_full.sum() / mask_full.sum()
print(f"Average loss (full): {total_full:.4f}")
print("→ Every token in thinking and answer is optimized")
print()

# ============================================================
# Strategy B: Answer Only — only tokens after </think>
# ============================================================
print("=" * 60)
print("Strategy B: Answer Only (only answer section)")
print("=" * 60)

def make_answer_only_mask(labels, think_start=100, think_end=101, ignore_id=0):
    """
    Only let tokens after </think> contribute to loss.
    Logic: find the </think> position in each sequence; tokens after it count.
    """
    mask = torch.zeros_like(labels, dtype=torch.float)

    for b in range(labels.shape[0]):
        # Find </think> position
        end_positions = (labels[b] == think_end).nonzero(as_tuple=True)[0]
        if len(end_positions) > 0:
            end_pos = end_positions[0].item()
            # After </think> (not including it) to before EOS/PAD
            for pos in range(end_pos + 1, labels.shape[1]):
                if labels[b, pos] == ignore_id:
                    break
                mask[b, pos] = 1.0
        # If no think_end found → entire sequence may be non-thinking mode → count all

    return mask

mask_answer_only = make_answer_only_mask(batch_labels)
print("loss_mask (0=excluded, 1=included):")
print(mask_answer_only.int())
print()

# Label which tokens belong to thinking, which to answer
print("Token labels (T=thinking, A=answer, S=special, P=PAD):")
for b in range(2):
    row = ""
    for pos in range(16):
        tid = batch_labels[b, pos].item()
        if tid == 0:
            row += " P "
        elif tid == 100:
            row += "[S "
        elif tid == 101:
            row += " S]"
        elif mask_answer_only[b, pos] == 1:
            row += " A "
        else:
            row += " T "
    print(f"  Sample{b}: {row}")
print()

# Construct masked labels: positions not in loss set to IGNORE
labels_masked_B = batch_labels.clone()
labels_masked_B[mask_answer_only == 0] = IGNORE

loss_answer_only = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    labels_masked_B.view(-1),
    ignore_index=IGNORE,
)
print(f"Average loss (answer only): {loss_answer_only:.4f}")
print("→ Only answer tokens are optimized, thinking is free-form")
print()

# ============================================================
# Strategy C: Selective Weighting — thinking down-weighted
# ============================================================
print("=" * 60)
print("Strategy C: Selective Weighting (thinking ×0.1)")
print("=" * 60)

def make_selective_weight_mask(labels, think_start=100, think_end=101,
                                thinking_weight=0.1, ignore_id=0):
    """
    Both thinking and answer tokens contribute to loss, but with different weights:
    - thinking token: weight = 0.1
    - answer token:   weight = 1.0
    - special token:  excluded
    """
    weight_mask = torch.zeros_like(labels, dtype=torch.float)

    for b in range(labels.shape[0]):
        in_thinking = False
        for pos in range(labels.shape[1]):
            tid = labels[b, pos].item()

            if tid == think_start:
                in_thinking = True
                continue  # special token itself excluded
            elif tid == think_end:
                in_thinking = False
                continue  # special token itself excluded
            elif tid == ignore_id:
                break  # PAD excluded

            if in_thinking:
                weight_mask[b, pos] = thinking_weight  # thinking: 0.1
            else:
                weight_mask[b, pos] = 1.0               # answer: 1.0

    return weight_mask

weight_mask = make_selective_weight_mask(batch_labels)
print("Weight matrix:")
for b in range(2):
    print(f"  Sample{b}: {[f'{w:.1f}' for w in weight_mask[b].tolist()]}")
print()

# Weighted loss
loss_per_token = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    batch_labels.view(-1),
    ignore_index=0,
    reduction='none'
).view(2, 16)

weighted_loss = (loss_per_token * weight_mask).sum() / weight_mask.sum()
print(f"Average loss (selective weighting): {weighted_loss:.4f}")
print("→ thinking gets 10% optimization signal, answer gets 100%")

# ============================================================
# Three-strategy comparison
# ============================================================
print(f"\n{'='*60}")
print("Three-strategy comparison summary")
print("=" * 60)
print(f"""
  ┌──────────────────┬──────────┬──────────┬──────────┐
  │                  │ Full Loss│Answer Only│Selective │
  ├──────────────────┼──────────┼──────────┼──────────┤
  │ Avg Loss         │ {total_full:.4f}   │ {loss_answer_only:.4f}   │ {weighted_loss:.4f}   │
  │ thinking optim.  │    ✅    │    ❌    │   🔶0.1  │
  │ answer optim.    │    ✅    │    ✅    │    ✅    │
  │ thinking stable  │    ✅    │    ❌    │    ✅    │
  │ complexity       │   low    │  medium  │  medium  │
  └──────────────────┴──────────┴──────────┴──────────┘
""")

print("How to choose in real training?")
print("  SFT phase → commonly use Answer Only (let the model learn the format first)")
print("  RL phase → commonly use Full Loss (DeepSeek-R1 approach)")
print("  Hybrid phase → use Selective Weighting for smooth transition")
print()
print("Common to all strategies: the special tokens themselves (<think>/</think>) do not count in loss.")
print("Because these tokens are just markers, their 'prediction' carries no meaning.")


## 4. How to Train a Thinking Model

Training a thinking model has four steps:

```
Step 1: Cold-start SFT
  Collect a few thousand high-quality examples "with a thinking process"
  Format: Q → thinking... → A
  Use these data for supervised fine-tuning, teaching the model the "think before answering" format

Step 2: RL reasoning training (the core!)
  Use reinforcement learning to train the model's reasoning ability
  Reward signals:
    - correct answer → +1
    - wrong answer → -1
    - correct format (has thinking tags) → +0.1
    - language consistency (thinking and answer in same language) → +0.1
  The model explores better reasoning paths on its own

Step 3: Rejection sampling + SFT
  Use the trained model to generate a large amount of question→thinking→answer data
  Only keep samples with correct answers
  Use this high-quality data for another round of SFT

Step 4: Full-scenario RL
  Do RL on more types of data (helpfulness, safety, etc.)
  So the model not only reasons well, but also converses naturally
```

**The most critical step is Step 2**: RL lets the model explore reasoning strategies on its own, rather than memorizing human reasoning processes.


In [ ]:
# Simulating the role of reward signals in RL training
print("=== RL Reasoning Training Simulation ===")
print()

print("Question: 15 + 27 = ?")
print()

# Simulate the model trying different reasoning paths
attempts = [
    ("15+20=35, 35+7=42", 42, True),
    ("15+27=42", 42, True),
    ("15+30=45, 45-3=42", 42, True),
    ("direct guess: 41", 41, False),
    ("10+20=30, 5+7=12, 30+12=42", 42, True),
]

for i, (reasoning, answer, correct) in enumerate(attempts):
    reward = 1 if correct else -1
    # Extra reward: detailed reasoning steps
    steps = len(reasoning.split(','))
    detail_bonus = min(steps * 0.05, 0.2)
    total_reward = reward + detail_bonus

    status = '✅' if correct else '❌'
    print(f"Attempt {i+1}: {status} {reasoning}")
    print(f"  answer={answer}, correct={correct}, base_reward={reward}, detail_bonus={detail_bonus:.2f}")
    print(f"  total_reward={total_reward:+.2f}")
    print()

print("RL reinforces high-reward reasoning patterns and suppresses low-reward ones.")
print("The model gradually learns: more detailed and correct reasoning → higher reward.")


## 5. Training Your Own Thinking Model

You do not need to start from scratch. Fine-tune an open-source model. Full process:

```
1. Choose a base model
   → Qwen2.5-7B / DeepSeek-V3 / Llama-3 etc.
   → Requirement: the base model itself must have decent reasoning ability

2. Prepare cold-start data (a few thousand examples for a teaching experiment; real effect depends on quality and coverage)
   → Use a strong model to generate checkable "question→steps→answer" data; note that reasoning APIs may not allow exporting hidden reasoning
   → Or extract from datasets like GSM8K, MATH
   → Format:
     <|user|>357 × 289 = ?
     <|assistant|><think>
     357×200=71400
     357×80=28560
     357×9=3213
     71400+28560+3213=103173
     </think>
     103173

3. Cold-start SFT (a teaching experiment can run LoRA at small scale; time depends on model, sequence length, and hardware)
   → Use tools like LLaMA-Factory / Axolotl
   → Teach the model the <think>/answer format

4. RL training (an important part of the R1-style route; time depends on rollout count, problem count, average generation length, and hardware)
   → Use frameworks like verl / OpenRLHF
   → Reward function: correct answer + correct format
   → Math problems use rule-based verification (answer correctness is clear-cut)
   → Code problems use test cases for verification

5. Rejection sampling + second round of SFT
   → Use the trained model to generate more data
   → Filter out wrong answers, keep correct ones
   → Do another round of SFT to consolidate
```

**Cost estimate**: Taking Qwen2.5-7B-scale as a reference, the full cost cannot be summarized with a fixed price; you must first determine the number of problems, samples per problem, average generation length, training epochs, and GPU/API unit price.


In [ ]:
# Simulating cold-start data format
print("=== Cold-Start Data Format Examples ===")
print()

training_examples = [
    {
        "question": "A rectangle has length 12cm and width 8cm. Find its area.",
        "thinking": "Rectangle area = length × width\nArea = 12 × 8 = 96\nSo the area is 96 square centimeters.",
        "answer": "96 square centimeters"
    },
    {
        "question": "Xiaoming has 15 apples. He eats 3, then buys 7 more. How many does he have now?",
        "thinking": "Start: 15\nEats 3: 15 - 3 = 12\nBuys 7 more: 12 + 7 = 19\nSo he now has 19 apples.",
        "answer": "19"
    },
    {
        "question": "357 × 289 = ?",
        "thinking": "357 × 200 = 71400\n357 × 80 = 28560\n357 × 9 = 3213\n71400 + 28560 + 3213 = 103173",
        "answer": "103173"
    }
]

for i, ex in enumerate(training_examples):
    print(f"--- Sample {i+1} ---")
    print(f"<|user|>\n{ex['question']}\n")
    print(f"<|assistant|><think>\n{ex['thinking']}\n</think>\n{ex['answer']}")
    print()

print("This is the format the model learns during SFT.")
print("During RL, the model explores better thinking content on its own.")


## 6. The Aha Moment of Thinking Models

The DeepSeek-R1 paper mentions an interesting phenomenon: during RL training, the model spontaneously learned to "reflect."

```
Model thinking process:
  Step 1: I think the answer is 42...
  Step 2: Wait, let me recheck...
  Step 3: Oh no, 15+27 should be 42, but my reasoning just now had a problem...
  Step 4: Recalculate: 15+20=35, 35+7=42. Confirmed, the answer is 42.
```

The DeepSeek-R1 paper observed that R1-Zero exhibited a similar "recheck" aha moment during RL. A more conservative reading is: on verifiable tasks, the reward signal may encourage the model to explore longer, more self-checking reasoning trajectories; this is not guaranteed, nor does it mean the model understands itself the way a human does.


In [ ]:
# Simulating the emergence of reflection behavior in RL training
print("=== Emergence of Reflection Behavior in RL Training ===")
print()

def evaluate_reasoning(reasoning, correct_answer):
    """Evaluate the reasoning process, returns (answer, is_correct, reward)"""
    import re
    numbers = re.findall(r'[\d.]+', reasoning)
    answer = float(numbers[-1]) if numbers else None
    correct = (answer == correct_answer)

    reward = 1.0 if correct else -1.0
    steps = reasoning.count('+') + reasoning.count('-') + reasoning.count('×') + reasoning.count('=')
    reward += min(steps * 0.05, 0.2)
    has_check = any(w in reasoning for w in ['verify', 'check', 'confirm', 'recalculate', 'wrong'])
    if has_check and correct:
        reward += 0.15
    return answer, correct, reward

correct_answer = 42.0
stages = [
    ("Early training", [
        "15+27=41",
        "15+27=44",
        "15+27=39",
    ]),
    ("Mid training (reflection begins)", [
        "15+27=41... wrong, recalculate. 15+20=35, 35+7=42. Verify: 42-27=15 ✓",
        "15+20=35, 35+7=42",
        "10+20=30, 5+7=12, 30+12=42. Confirmed correct.",
    ]),
    ("Late training (stable reasoning)", [
        "15+20=35, 35+7=42. Verify: 42-27=15 ✓",
        "First compute 15+20=35, then add 7=42. Check: 42-15=27 ✓",
        "15+27: split into 15+20=35, 35+7=42. Verify 42-27=15, correct.",
    ]),
]

for stage_name, outputs in stages:
    print(f"📋 {stage_name}:")
    total_reward = 0
    for reasoning in outputs:
        answer, correct, reward = evaluate_reasoning(reasoning, correct_answer)
        status = '✅' if correct else '❌'
        total_reward += reward
        print(f"  {status} {reasoning[:60]}...")
        print(f"     answer={answer}, reward={reward:+.2f}")
    avg_reward = total_reward / len(outputs)
    print(f"  Average reward: {avg_reward:+.2f}")
    print()

print("Trend: early guessing → mid-stage occasional reflection (high reward) → late-stage reflection becomes habitual")
print("This is emergent behavior: no one taught the model to reflect; the RL reward made it discover this on its own.")


## 7. Limitations of Thinking Models

| Limitation | Description |
|:---|:---|
| **Slow** | The thinking process can be long (hundreds to thousands of tokens); the user has to wait |
| **Expensive** | Thinking tokens also cost money (APIs charge by token) |
| **Overthinking** | Even simple questions get lengthy reasoning — "1+1=?" may trigger 500 words of reasoning |
| **Language mixing** | Thinking may mix languages, hurting readability |
| **Uncontrollable** | RL-trained thinking strategies are a black box, not necessarily matching human expectations |

**Best suited for**: math, programming, logical reasoning, multi-constraint planning, scientific derivation.
**Not necessarily suited for**: chitchat, short Q&A, stylistic writing, emotional companionship, low-latency scenarios. The reason is not that thinking is useless, but that the extra reasoning tokens add latency and cost, and may cause overthinking.


## 8. Comparison of Mainstream Thinking Models

| Model | Training Method | Key Features |
|:---|:---|:---|
| **OpenAI o-series** | Training details not public | Closed-source reasoning models; API provides reasoning effort / reasoning configuration |
| **DeepSeek-R1** | Paper fairly detailed | The public paper describes a multi-stage pipeline: cold-start, RL, rejection sampling, SFT, etc. |
| **Qwen3** | Official docs support thinking/non-thinking | Supports switching thinking behavior via chat template / parameters |
| **Kimi / other reasoning models** | Public info varies by version | Only write specific training methods when there is an official technical report or model card |

**Common thread**: all distinguish "thinking/reasoning budget" from "final answer" at the product or model-behavior level, but the training details of closed-source models cannot be stated as established facts.


## 9. Practice: Enabling and Switching Thinking Modes

Each platform enables thinking differently, and these APIs change with model versions. What follows is the most reliable usage based on currently available documentation. Remember to "check the model docs" rather than memorizing any single parameter as permanent.

---

#### 9.1 OpenAI reasoning models (o-series, GPT-5-series)

OpenAI reasoning models typically control thinking intensity via `reasoning_effort` or the `reasoning` configuration in the Responses API:

```text
Common values: "low" | "medium" | "high"
Some newer models also support "minimal" or "none". Refer to the official model docs.
```

**Key points**:
- The models that support reasoning are OpenAI reasoning models, such as o3, o4-mini, GPT-5-series, etc.
- Regular models like GPT-4o should not be treated as reasoning models by default.
- The API typically does not return the full hidden thinking process, but may return reasoning token usage.
- Billing and context usage must account for reasoning tokens.

Reference: [OpenAI reasoning guide](https://platform.openai.com/docs/guides/reasoning), [OpenAI models](https://platform.openai.com/docs/models).

---

#### 9.2 DeepSeek-R1 / DeepSeek thinking mode

Open-source thinking models like DeepSeek-R1 commonly use a **Chat Template** to separate the thinking section from the final answer. Some frontends collapse the model's `<think>...</think>` section:

```text
Model output format:
  <think>
  First analyze the problem conditions...
  Check if the calculation is correct...
  </think>
  42

Frontend handling:
  - By default, collapse content between <think>...</think>
  - Only display the final answer after </think>
```

**A common misconception**: you cannot simply say "DeepSeek can never turn off thinking." Local R1 / R1-distill models will naturally output thinking format; but DeepSeek's official API now supports enabling or disabling thinking via parameters.

Common scenarios:
- `deepseek-reasoner` returns `reasoning_content` and the final `content`.
- Newer DeepSeek Chat/Reasoner APIs support switches like `thinking.enabled`; refer to the official docs for specifics.

Reference: [DeepSeek reasoning model](https://api-docs.deepseek.com/guides/reasoning_model), [DeepSeek thinking mode](https://api-docs.deepseek.com/guides/thinking_mode).

---

#### 9.3 Qwen3 (explicitly supports Thinking / Non-Thinking dual mode)

Qwen3 is one of the few mainstream open-source models that explicitly supports thinking/non-thinking switching in its official chat template:

```text
Method 1: via chat template parameter
  enable_thinking=True   → enter thinking mode
  enable_thinking=False  → disable thinking, answer directly

Method 2: via commands in the conversation
  user: /think       → enable thinking
  user: /no_think    → disable thinking
```

**How it works**: Qwen3 was trained on both thinking and non-thinking data, so it can switch behavior based on the `enable_thinking` toggle.

**Practical usage** (HuggingFace Transformers):
```python
# Enable thinking
messages = [{"role": "user", "content": "357×289=?"}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False,
    enable_thinking=True
)

# Disable thinking
text = tokenizer.apply_chat_template(
    messages, tokenize=False,
    enable_thinking=False
)
```

Reference: [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B), [Qwen quickstart](https://qwen.readthedocs.io/en/v3.0/getting_started/quickstart.html).

---

#### 9.4 Anthropic Claude (Extended Thinking)

Claude's Extended Thinking also has version differences. Claude 3.7/4.x used to support manually setting `budget_tokens`, but newer Opus 4.7/4.8 documentation recommends adaptive thinking and no longer supports manual `budget_tokens`. So check the model version before writing code.

The old-style manual budget form looks like this:

```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=20000,
    thinking={
        "type": "enabled",
        "budget_tokens": 4000
    },
    messages=[{"role": "user", "content": "Prove that the square root of 2 is irrational"}]
)
```

Newer models may use `effort` or adaptive thinking. Reference: [Anthropic extended thinking docs](https://platform.claude.com/docs/en/build-with-claude/extended-thinking).

---

#### 9.5 Local inference (vLLM / SGLang / Ollama)

If deploying R1 distill models with vLLM or SGLang, thinking behavior is typically controlled by the **chat template** and the inference framework's parser:

```bash
# vLLM with DeepSeek-R1-Distill-Qwen-7B
vllm serve deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
  --enable-reasoning \
  --reasoning-parser deepseek_r1
```

The frontend can choose to display, collapse, or filter `<think>...</think>`. This is not about the model "not thinking," but about the presentation layer deciding whether to show thinking content to the user.

---

#### Summary: Enabling Thinking across Platforms

| Platform/model | How to enable | Can disable? | Thinking content visible? |
|:---|:---|:---|:---|
| OpenAI reasoning models | `reasoning_effort` / `reasoning` config | Adjustable intensity; values vary by model | Usually not visible; only token usage shown |
| DeepSeek API | `deepseek-reasoner` or `thinking.enabled` | Newer API configurable; local R1 depends on template | API can return `reasoning_content` |
| Qwen3 | `enable_thinking=True/False`, `/think`, `/no_think` | Switchable | Template controls visibility |
| Claude | Extended Thinking / adaptive thinking | Depends on model version | Returned in separate blocks or per-version policy |
| Local R1 distill | Chat template + parser | Model behavior mostly fixed; frontend can filter | Can display or collapse |


In [ ]:
# ============================================================
# Practice: API calling examples for each platform (real runnable code)
# Note: requires environment variable API keys for actual calls
# If no key is set, prints equivalent curl command and skips
# ============================================================

import os

# ----------------------------------------------------------
# 1. OpenAI o3 API call
# ----------------------------------------------------------
openai_key = os.environ.get("OPENAI_API_KEY")
if openai_key:
    from openai import OpenAI
    client = OpenAI(api_key=openai_key)

    response = client.chat.completions.create(
        model="o3-mini",
        reasoning_effort="medium",  # low | medium | high
        messages=[
            {"role": "user", "content": "357 x 289 = ?"}
        ]
    )

    print("=== OpenAI o3-mini Result ===")
    print(f"Answer: {response.choices[0].message.content}")
    print(f"Token usage: {response.usage}")
else:
    print("=== OpenAI o3 API (OPENAI_API_KEY not set, skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.openai.com/v1/chat/completions \\')
    print('  -H "Content-Type: application/json" \\')
    print('  -H "Authorization: Bearer $OPENAI_API_KEY" \\')
    print("  -d '{\"model\":\"o3-mini\",\"reasoning_effort\":\"medium\",\"messages\":[{\"role\":\"user\",\"content\":\"357x289=?\"}]}'")

# ----------------------------------------------------------
# 2. DeepSeek-R1 API call
# ----------------------------------------------------------
deepseek_key = os.environ.get("DEEPSEEK_API_KEY")
if deepseek_key:
    from openai import OpenAI
    client = OpenAI(
        api_key=deepseek_key,
        base_url="https://api.deepseek.com"
    )

    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[
            {"role": "user", "content": "357 x 289 = ?"}
        ]
    )

    print("\n=== DeepSeek-R1 Result ===")
    rc = response.choices[0].message.reasoning_content
    print(f"Thinking process: {rc[:200]}...")
    print(f"Final answer: {response.choices[0].message.content}")
else:
    print("\n=== DeepSeek-R1 API (DEEPSEEK_API_KEY not set, skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.deepseek.com/chat/completions \\')
    print('  -H "Content-Type: application/json" \\')
    print('  -H "Authorization: Bearer $DEEPSEEK_API_KEY" \\')
    print("  -d '{\"model\":\"deepseek-reasoner\",\"messages\":[{\"role\":\"user\",\"content\":\"357x289=?\"}]}'")

# ----------------------------------------------------------
# 3. Qwen3 local inference (HuggingFace Transformers + thinking toggle)
# ----------------------------------------------------------
QWEN3_MODEL = "Qwen/Qwen3-8B"
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    print("\n=== Qwen3 Local Inference ===")
    tokenizer = AutoTokenizer.from_pretrained(QWEN3_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        QWEN3_MODEL,
        torch_dtype="auto",
        device_map="auto"
    )

    messages = [{"role": "user", "content": "357 x 289 = ?"}]

    # --- Method A: enable thinking ---
    print("\n>>> enable_thinking=True (thinking mode enabled)")
    text_on = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=True
    )
    inputs = tokenizer(text_on, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024)
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"Output (with thinking tags): {result[:300]}...")

    # --- Method B: disable thinking ---
    print("\n>>> enable_thinking=False (thinking disabled, answer directly)")
    text_off = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False
    )
    inputs = tokenizer(text_off, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512)
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"Output: {result[:300]}...")

except Exception as e:
    print(f"\n=== Qwen3 Local Inference (skipped: {type(e).__name__}) ===")
    print(f"Reason: {e}")
    print()
    print("To run Qwen3 locally, execute:")
    print("  pip install transformers torch accelerate")
    print(f'  model = AutoModelForCausalLM.from_pretrained("{QWEN3_MODEL}", device_map="auto")')
    print("  tokenizer.apply_chat_template(messages, enable_thinking=True)")

# ----------------------------------------------------------
# 4. Claude Extended Thinking API call
# ----------------------------------------------------------
anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
if anthropic_key:
    import anthropic
    client = anthropic.Anthropic(api_key=anthropic_key)

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=20000,
        thinking={
            "type": "enabled",
            "budget_tokens": 4000
        },
        messages=[
            {"role": "user", "content": "Prove that the square root of 2 is irrational"}
        ]
    )

    print("\n=== Claude Extended Thinking Result ===")
    for block in response.content:
        if block.type == "thinking":
            print(f"Thinking process: {block.thinking[:200]}...")
        elif block.type == "text":
            print(f"Final answer: {block.text[:200]}...")
else:
    print("\n=== Claude API (ANTHROPIC_API_KEY not set, skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.anthropic.com/v1/messages \\')
    print('  -H "x-api-key: $ANTHROPIC_API_KEY" \\')
    print('  -H "anthropic-version: 2023-06-01" \\')
    print('  -H "content-type: application/json" \\')
    print("  -d '{\"model\":\"claude-sonnet-4-20250514\",\"max_tokens\":20000,\"thinking\":{\"type\":\"enabled\",\"budget_tokens\":4000},\"messages\":[{\"role\":\"user\",\"content\":\"Prove sqrt(2) is irrational\"}]}'")

# ----------------------------------------------------------
# 5. Via OpenRouter (OpenAI-compatible API, supports multiple models)
# ----------------------------------------------------------
openrouter_key = os.environ.get("OPENROUTER_API_KEY")
if openrouter_key:
    from openai import OpenAI
    client = OpenAI(
        api_key=openrouter_key,
        base_url="https://openrouter.ai/api/v1"
    )

    response = client.chat.completions.create(
        model="deepseek/deepseek-r1",
        messages=[
            {"role": "user", "content": "What is entropy? Explain in one sentence."}
        ]
    )

    print("\n=== OpenRouter (DeepSeek-R1) Result ===")
    msg = response.choices[0].message
    reasoning = getattr(msg, "reasoning", None)
    if reasoning:
        print(f"Thinking process: {reasoning[:200]}...")
    print(f"Final answer: {msg.content[:200]}...")
else:
    print("\n=== OpenRouter API (OPENROUTER_API_KEY not set, skipping) ===")
    print("Register for a key: https://openrouter.ai/keys")
    print("Supported models: deepseek/deepseek-r1, openai/o1, anthropic/claude-sonnet-4, etc.")
    print()
    print("Equivalent curl:")
    print('curl https://openrouter.ai/api/v1/chat/completions \\')
    print('  -H "Authorization: Bearer $OPENROUTER_API_KEY" \\')
    print('  -H "Content-Type: application/json" \\')
    print("  -d '{\"model\":\"deepseek/deepseek-r1\",\"messages\":[{\"role\":\"user\",\"content\":\"What is entropy?\"}]}'")


## 10. Hands-on: Training a Thinking Model

Below is a **teaching-version full pipeline**, based on Qwen2.5-7B, with the goal of understanding how a "think before answering" reasoning model is trained. It is suitable for small-scale experiments; if you want to train something close to DeepSeek-R1/Qwen3-level capability, the data scale, RL sampling volume, reward design, and compute will all be much higher.

---

#### Step 1: Environment Setup

```bash
# Create virtual environment
conda create -n thinking-train python=3.10 -y
conda activate thinking-train

# Install core dependencies
pip install torch==2.4.0 transformers datasets accelerate
pip install vllm  # For efficient inference and generating training data

# Install training framework (choose one)
pip install llama-factory  # Recommended for beginners, has Web UI
# or
pip install axolotl        # More flexible, for advanced users
```

---

#### Step 2: Prepare Cold-start Data (~3000-5000 examples)

The core of cold-start data is the **"question → thinking process → answer"** triple.

**Data sources**:
1. Use a strong model to generate checkable solution steps; if you use a reasoning API, follow that platform's policy on outputting hidden reasoning
2. Extract from public datasets: GSM8K (elementary math), MATH (competition math), APPS (programming)
3. Mix Chinese and English data so the model does not bias toward one language

**Data format** (Qwen3 / DeepSeek style):
```json
[
  {
    "messages": [
      {"role": "user", "content": "A water tank: the inlet pipe fills it in 3 hours, the outlet pipe drains it in 5 hours. With both open, how long to fill?"},
      {"role": "assistant", "content": "<think>\nInflow rate = 1/3 tank/hour\nOutflow rate = 1/5 tank/hour\nNet rate = 1/3 - 1/5 = 2/15 tank/hour\nTime to fill = 1/(2/15) = 15/2 = 7.5 hours\n</think>\n7.5 hours"}
    ]
  }
]
```

**Important**: the thinking process should be detailed but not excessive. For simple problems, write 50-100 words; for hard problems, 200-500 words. If every "1+1=?" gets 500 words of reasoning, the trained model will have a severe overthinking problem.

---

#### Step 3: Cold-start SFT (Teach the Model the Format)

This is the critical step that teaches the model the `<think>` / answer format.

**Using LLaMA-Factory** (recommended, has GUI so fewer mistakes):

```bash
# Start LLaMA-Factory Web UI
llamafactory-cli webui

# In the Web UI:
# 1. Select model: Qwen/Qwen2.5-7B-Instruct
# 2. Upload your cold-start dataset
# 3. Training type: Supervised Fine-Tuning
# 4. LoRA: rank=64, alpha=128
# 5. Learning rate: 5e-5, epochs: 3
# 6. Sequence length: 4096
# 7. batch_size: 4, gradient_accumulation: 4
# 8. Start training!
```

**Using command line**:

```bash
llamafactory-cli train \
    --model_name_or_path Qwen/Qwen2.5-7B-Instruct \
    --dataset my_cot_coldstart \
    --template qwen \
    --finetuning_type lora \
    --lora_rank 64 \
    --lora_alpha 128 \
    --output_dir ./output/qwen-sft-cot \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --save_steps 500 \
    --learning_rate 5e-5 \
    --num_train_epochs 3 \
    --bf16
```

**Expected result**: after training, the model will write thinking for every question, but quality depends on the quality of the cold-start data.

---

#### Step 4: RL Reasoning Training (the Core — Making the Model Smarter)

SFT mainly makes the model learn the format and basic problem-solving style; RL is one of the key parts of the R1-style route, but data, base model, reward function, and sampling scale are equally important.

**Using verl** (one of the open-source frameworks commonly used for RLHF/RL training experiments):

```bash
# Install verl
git clone https://github.com/volcengine/verl.git
cd verl
pip install -e .

# Prepare reward function dataset: only need (question, correct_answer)
# Format: {"prompt": "...", "answer": "..."}
```

**verl config file** (`r1_train.yaml`):

```yaml
# Model config
actor_rollout_ref:
  model:
    path: ./output/qwen-sft-cot  # Model after cold-start
    use_fused_kernels: true
  rollout:
    n: 4  # Generate 4 candidate answers per prompt
    temperature: 1.0

# Reward functions
reward_fn:
  # Math problems: compare model output with the standard answer
  - name: math_verify
    weight: 0.7
  # Format check: ensure thinking tags are present
  - name: format_check
    weight: 0.2
  # Language consistency
  - name: language_consistency
    weight: 0.1

# Training config
trainer:
  n_gpus_per_node: 4
  nnodes: 1
  total_epochs: 1
  project_name: "my-thinking-model"
```

```bash
# Start RL training
python -m verl.trainer.main_ppo \
    --config-name r1_train.yaml
```

**Using OpenRLHF** (alternative, easier to get started):

```bash
# Install
git clone https://github.com/OpenRLHF/OpenRLHF.git
cd OpenRLHF
pip install -e .

# Ray cluster mode (4 GPUs)
ray start --head --num-gpus=4

python -m openrlhf.cli.train_ppo_ray \
    --ref_num_nodes 1 \
    --ref_num_gpus_per_node 2 \
    --reward_num_nodes 1 \
    --reward_num_gpus_per_node 1 \
    --actor_num_nodes 1 \
    --actor_num_gpus_per_node 1 \
    --pretrain ./output/qwen-sft-cot \
    --reward_fn math_verify \
    --save_path ./output/qwen-rl \
    --prompt_data ./data/math_prompts.jsonl
```

**Key tips for RL training**:

| Tip | Description |
|:---|:---|
| **Math problems are the best RL starting point** | Answer correctness is clear-cut, no manual reward annotation needed |
| **Format reward should not be too high** | Too high makes the model focus only on format, not correctness |
| **High temperature (0.7-1.0)** | RL needs exploration; low temperature suppresses discovery of new strategies |
| **Generate 4-8 candidates per prompt** | Sufficient sampling is needed to discover good reasoning paths |
| **The reward function is your "syllabus"** | What you reward, the model learns |

---

#### Step 5: Rejection Sampling + Second Round of SFT

```bash
# Use the trained model to generate answers for 5000 new questions
# Only keep samples with correct answers ("reject" wrong ones)
python scripts/rejection_sampling.py \
    --model ./output/qwen-rl \
    --prompts ./data/new_prompts.jsonl \
    --output ./data/rl_filtered.jsonl \
    --num_samples 4 \
    --keep_correct_only

# Use filtered high-quality data for another round of SFT
llamafactory-cli train \
    --model_name_or_path ./output/qwen-rl \
    --dataset rl_filtered \
    --template qwen \
    --finetuning_type lora \
    --output_dir ./output/qwen-final \
    --learning_rate 1e-5 \
    --num_train_epochs 2
```

---

#### Cost and Time Estimates

| Step | Hardware | Time | Cost (cloud GPU) |
|:---|:---|:---|:---|
| Cold-start SFT | 1×A100 (80G) | Hours | Depends on cloud provider pricing |
| RL reasoning training | Usually requires multiple GPUs or high-throughput inference resources | Possibly hours to days | Strongly depends on rollout count, problem count, sampling length, and parallelism |
| Rejection sampling + SFT | 1×A100 (80G) | Hours | Strongly depends on the number of generated samples |

> Do not memorize a fixed price here. A small-scale math-problem experiment might run the pipeline for a few hundred dollars; building a stable, generalizable thinking model usually costs significantly more. To estimate real costs, first determine the number of problems, samples per problem, average generation length, training epochs, and GPU unit price.

---

#### Common Pitfalls for Beginners

1. **Thinking too short in cold-start data** → model learns the format but cannot reason → fix: ensure thinking is at least 50+ words
2. **Reward function only rewards correctness** → model learns to cheat (e.g., outputs the answer directly without thinking) → fix: add a format reward
3. **Math dataset too easy** → RL converges quickly but reasoning ability does not improve → fix: mix at least 30% hard problems
4. **Training set and test set identical** → model memorizes rather than learning to reason → fix: strictly split train/test by problem ID
5. **Wrong chat template** → thinking tags render incorrectly, model behavior is abnormal → fix: verify against the official model documentation template

---

#### Easy Route: Use R1 Distill Models Without Training

If you only want to **use** a thinking model without training, directly use the distill models released by DeepSeek:

```bash
# DeepSeek-R1 distill versions (already have thinking ability, no training needed)
# 1.5B version: suitable for toy projects
# 7B version:   suitable for research and experiments
# 14B version:  reasoning ability noticeably improved
# 32B version:  close to the original R1 reasoning level
# 70B version:  strongest distill version

# Download and use directly
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# This model already knows how to think! Format is identical to R1.
# If you want to "disable thinking," you can fine-tune on non-thinking data based on this.
```

Distill models are essentially "student models" of R1 — trained using R1's thinking data.
Their reasoning ability is far stronger than regular models of the same size, but not as strong as the original R1.


In [ ]:
# ============================================================
# Practice: Training scripts (all functions are complete runnable implementations)
# ============================================================

import json
import os

import re

# ----------------------------------------------------------
# A. Cold-start data generation (using DeepSeek API to generate training data)
# ----------------------------------------------------------
def generate_coldstart_data(questions, output_path="coldstart_data.json"):
    """Use the DeepSeek-R1 API to batch-generate question-thinking-answer training data"""
    deepseek_key = os.environ.get("DEEPSEEK_API_KEY")
    if not deepseek_key:
        print("SKIP: DEEPSEEK_API_KEY environment variable required")
        return []

    from openai import OpenAI
    client = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com")

    SYSTEM_PROMPT = """You are a math reasoning assistant. For each question:
1. First write out a detailed step-by-step reasoning process
2. Explain the calculation logic clearly at each step
3. Finally give a concise answer

Important: the output format must be:
<think>
(reasoning process)
</think>
(final answer)
"""

    dataset = []
    for i, question in enumerate(questions):
        print(f"[{i+1}/{len(questions)}] {question[:50]}...")
        try:
            response = client.chat.completions.create(
                model="deepseek-reasoner",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question}
                ]
            )
            thinking = response.choices[0].message.reasoning_content or ""
            answer = response.choices[0].message.content or ""

            content = f"<think>\n{thinking}\n</think>\n{answer}"
            dataset.append({
                "messages": [
                    {"role": "user", "content": question},
                    {"role": "assistant", "content": content}
                ]
            })
        except Exception as e:
            print(f"  Failed: {e}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(dataset)} examples to {output_path}")
    return dataset

# Example question set
SAMPLE_QUESTIONS = [
    "A rectangle has length 12cm and width 8cm. Find its area.",
    "Xiaoming has 15 apples. He eats 3, then buys 7 more. How many does he have now?",
    "357 x 289 = ?",
    "A water tank: the inlet pipe fills it in 3 hours, the outlet pipe drains it in 5 hours. With both open, how long to fill?",
    "In the arithmetic sequence 3, 7, 11, ..., what is the 20th term?",
]

if os.environ.get("DEEPSEEK_API_KEY"):
    generate_coldstart_data(SAMPLE_QUESTIONS[:2])  # Demo 2 examples
else:
    print("=== generate_coldstart_data function defined ===")
    print("Set DEEPSEEK_API_KEY then call generate_coldstart_data(questions)")
    print("\nExample data structure:")
    example = {
        "messages": [
            {"role": "user", "content": "357 x 289 = ?"},
            {
                "role": "assistant",
                "content": (
                    "<think>\n"
                    "357 x 200 = 71400\n"
                    "357 x 80  = 28560\n"
                    "357 x 9   = 3213\n"
                    "71400 + 28560 + 3213 = 103173\n"
                    "</think>\n"
                    "103173"
                )
            }
        ]
    }
    print(json.dumps(example, ensure_ascii=False, indent=2))

# ----------------------------------------------------------
# B. RL Reward function (complete implementation, directly usable with verl/OpenRLHF)
# ----------------------------------------------------------
def extract_thinking_and_answer(completion):
    """Extract the thinking content and final answer from model output"""
    thinking = ""
    answer = completion.strip()

    think_match = re.search(r"<think>(.*?)</think>", completion, re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        answer = completion[think_match.end():].strip()

    return thinking, answer

def normalize_number(text):
    """Extract and normalize numbers from text, used for answer comparison"""
    text = text.strip().replace(" ", "").replace(",", "")
    match = re.search(r"-?[\d.]+(?:/-?[\d.]+)?", text)
    if match:
        num_str = match.group()
        if "/" in num_str:
            parts = num_str.split("/")
            try:
                return str(float(parts[0]) / float(parts[1]))
            except (ValueError, ZeroDivisionError):
                return num_str
        try:
            return str(float(num_str))
        except ValueError:
            return num_str
    return text

def compute_reward(completion, ground_truth):
    """
    Compute the reward for one model output.
    Reward components: correct format +0.15 | correct answer +1.0 | thinking length +0.05
    Penalties: severely wrong format -1.0 | wrong answer -0.5
    """
    reward = 0.0
    thinking, answer = extract_thinking_and_answer(completion)

    has_thinking = "<think>" in completion
    has_answer_tag = "</think>" in completion

    if not has_answer_tag:
        return -1.0

    if has_thinking and has_answer_tag:
        reward += 0.15

    pred_norm = normalize_number(answer)
    gt_norm = normalize_number(ground_truth)

    if pred_norm == gt_norm:
        reward += 1.0
    else:
        try:
            pv = float(pred_norm)
            gv = float(gt_norm)
            if abs(pv - gv) / max(abs(gv), 1e-8) < 0.001:
                reward += 1.0
            else:
                reward -= 0.5
        except ValueError:
            reward -= 0.5

    if len(thinking) >= 30:
        reward += 0.05

    return reward

# Test the reward function
print("=== RL Reward Function Test ===\n")
test_cases = [
    (
        "<think>15+20=35, 35+7=42</think>\n42",
        "42",
    ),
    (
        "<think>15+27=41</think>\n41",
        "42",
    ),
    (
        "No thinking process\nthe answer is 42",
        "42",
    ),
    (
        "<think>calculating</think>\n7.5",
        "7.5",
    ),
]

for completion, gt in test_cases:
    reward = compute_reward(completion, gt)
    print(f"Reward: {reward:+.2f} | Output fragment: {completion[:50]}...")
print()

# ----------------------------------------------------------
# C. Rejection sampling (complete implementation, directly usable for second-round SFT data preparation)
# ----------------------------------------------------------
def rejection_sampling(model, tokenizer, prompts, num_samples=4):
    """
    Generate num_samples candidate answers for each prompt,
    only keep those with correct answers, selecting the one with the most detailed thinking.

    Args:
        model: HuggingFace model instance
        tokenizer: HuggingFace tokenizer instance
        prompts: [{"question": "...", "answer": "..."}, ...]
        num_samples: how many candidates to generate per question
    Returns:
        Filtered training data list
    """
    import torch
    device = next(model.parameters()).device
    filtered_data = []
    total_candidates = 0
    total_correct = 0

    for i, prompt in enumerate(prompts):
        question = prompt["question"]
        ground_truth = prompt["answer"]
        candidates = []

        for _ in range(num_samples):
            messages = [{"role": "user", "content": question}]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    temperature=0.8,
                    do_sample=True,
                )

            completion = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            )

            thinking, answer = extract_thinking_and_answer(completion)
            is_correct = compute_reward(completion, ground_truth) > 0.5
            candidates.append((thinking, answer, is_correct))
            total_candidates += 1
            if is_correct:
                total_correct += 1

        correct = [(t, a) for t, a, ok in candidates if ok]
        if correct:
            best_t, best_a = max(correct, key=lambda x: len(x[0]))
            content = f"<think>\n{best_t}\n</think>\n{best_a}"
            filtered_data.append({
                "messages": [
                    {"role": "user", "content": question},
                    {"role": "assistant", "content": content}
                ]
            })

        if (i + 1) % 10 == 0:
            print(f"Progress: {i+1}/{len(prompts)} | "
                  f"candidates={total_candidates} correct={total_correct} "
                  f"kept={len(filtered_data)}")

    return filtered_data

print("=== rejection_sampling function defined ===")
print()
print("Usage:")
print("  from transformers import AutoModelForCausalLM, AutoTokenizer")
print('  model = AutoModelForCausalLM.from_pretrained("./output/qwen-rl")')
print("  prompts = [{'question': '...', 'answer': '...'}, ...]")
print("  new_data = rejection_sampling(model, tokenizer, prompts, num_samples=4)")
print()

# ----------------------------------------------------------
# D. Recommended directory structure
# ----------------------------------------------------------
print("=== Recommended Directory Structure ===")
print(r"""
my-thinking-model/
+-- data/
|   +-- coldstart.jsonl       # Cold-start SFT data (~3000 examples)
|   +-- rl_prompts.jsonl      # RL training question set (~1000 examples)
|   +-- rl_filtered.jsonl     # Data after rejection sampling
+-- configs/
|   +-- sft_coldstart.yaml    # LLaMA-Factory config
|   +-- rl_verl.yaml          # verl config
+-- output/
|   +-- qwen-sft-cot/         # Model after cold-start
|   +-- qwen-rl/              # Model after RL training
|   +-- qwen-final/           # Final model
+-- scripts/
|   +-- gen_coldstart.py      # Generate cold-start data (using the function above)
|   +-- rejection_sampling.py # Rejection sampling (using the function above)
|   +-- eval_thinking.py      # Evaluation script
+-- README.md
""")
print()
print("All functions above are complete runnable implementations (not pseudocode).")
print("Core idea: math problem answers can be automatically verified, making them the best data type for an RL starting point.")


## 11. The Other Path: Do Not Train the Model, Just Spend More Compute at Inference Time

The thinking models discussed earlier take the "training-side" route — using RL to make the model learn self-reflection. But there is another path: **do not change the model weights at all, just spend more compute at inference time**.

The earlier Self-Consistency (section 2.5) is actually one instance of this path: sample several chains and vote. This section looks at three other common methods; together they form the research direction of **test-time compute scaling** — the success of o1 is not just from training a thinking model, but also from using verifier-guided search at inference time.

The core differences between the four methods:

| Method | How many samples | How to pick the answer | Suitable scenario |
|:---|:---|:---|:---|
| Self-Consistency | N times | Majority vote | Discrete, comparable answers |
| Best-of-N + RM | N times | Reward Model scoring | Continuous, hard-to-compare answers |
| Tree of Thoughts | N × M times | Node evaluation + search | Multi-step decisions, with branching |
| Verifier-guided (PRM) | N times | Score every step | Math, code and other step-decomposable problems |
| Self-Refine / Reflexion | 1 + K times | Model self-eval + reflection | Open-ended generation |

Let us look at each one below.


### 11.1 Best-of-N + Reward Model

Self-Consistency picks the answer by "majority voting," but many tasks do not have discrete answers — writing a poem, writing a marketing copy, answering an open-ended question — there is no "standard answer" to vote on. This is where a **Reward Model (RM)** is needed to score each response.

Process:

1. Sample N responses for the same prompt (temperature > 0)
2. Use the RM to score each response
3. Pick the one with the highest score as the final output

Key difference:

- **Self-Consistency** is "majority voting," relying on answers being comparable
- **Best-of-N** is "quality scoring," relying on the RM's judgment ability

The ceiling of Best-of-N is determined by the RM. If the RM is weak, BoN cannot save it.


In [ ]:
# === Best-of-N + Reward Model simulation ===
# Demo: for the same open-ended question, sample 4 responses, use the RM to score and pick the best

import random
random.seed(42)

prompt = "Write one sentence praising a spring morning"

# Simulate: sample 4 responses for the same prompt (in reality generated by a model)
samples = [
    "A spring morning is truly beautiful: warm sunlight and blooming flowers.",
    "Dawn light filters through thin mist onto new grass, as if the whole winter had been crumbled and relaid.",
    "Spring morning is nice.",
    "In spring one sleeps unaware of dawn; everywhere one hears birds singing; in the night came wind and rain; who knows how many petals fell.",
]

# Simulate a Reward Model scoring (in reality, train an RM or use a strong model as a judge)
def mock_reward_model(prompt, response):
    """Simulate RM scoring: combine relevance, fluency, creativity"""
    score = 0.0
    # Bonus for moderate length
    if 15 < len(response) < 60:
        score += 2.0
    # Bonus for metaphor / imagery
    metaphors = ["like", "as if", "crumble", "filter", "through"]
    if any(m in response for m in metaphors):
        score += 3.0
    # Penalty for too short
    if len(response) < 10:
        score -= 2.0
    # Bonus for classical poem reference
    if "dawn" in response or "birds" in response:
        score += 1.5
    return round(score, 2)

# Run Best-of-N
print(f"=== Best-of-N (N=4) ===")
print(f"Prompt: {prompt}\n")

scored = []
for i, s in enumerate(samples, 1):
    score = mock_reward_model(prompt, s)
    scored.append((i, s, score))
    print(f"  Sample #{i}  score {score:>5}  | {s}")

# Pick the highest score
best = max(scored, key=lambda x: x[2])
print(f"\n{'=' * 60}")
print(f"🏆 Best-of-N picks: Sample #{best[0]}")
print(f"   Response: {best[1]}")
print(f"   Score: {best[2]}")
print(f"\nKey observation:")
print(f"  Sample #1 is ordinary but complete, scoring 2")
print(f"  Sample #2 has imagery and moderate length, scoring 5 (selected)")
print(f"  Sample #3 is too short, losing 2 points")
print(f"  Sample #4 references a classical poem but is too long, scoring 1.5")
print(f"  → The RM's judgment quality determines the ceiling of Best-of-N")


### 11.2 Tree of Thoughts: Search-Tree-Style Reasoning

CoT is a linear reasoning chain — if some step in the middle goes wrong, everything after is wrong. Tree of Thoughts (Yao et al. 2023) models reasoning as a **search tree**:

- Each node is an intermediate state (a partial reasoning result)
- From each node you can expand k candidate next steps
- An evaluator scores each candidate and decides which branch to take

Suitable scenarios: problems with clear intermediate states and the possibility of backtracking, such as the **24-game, crosswords, creative writing, and planning tasks**.

Cost: the compute is k^d times that of CoT (k is the branching factor, d is the depth), so it can only be used on problems worth spending compute on.


In [ ]:
# === Tree of Thoughts minimal demo: solve the 24-game with ToT ===
# Given 4 numbers, use + - × ÷ to get 24

from itertools import permutations, product

def solve_24_to(nums, ops_seq):
    """Try whether a (permutation, operation sequence) yields 24"""
    a, b, c, d = nums
    op1, op2, op3 = ops_seq
    op_map = {'+': lambda x, y: x + y, '-': lambda x, y: x - y,
              '×': lambda x, y: x * y, '÷': lambda x, y: x / y if y != 0 else None}
    try:
        r1 = op_map[op1](a, b)
        if r1 is None: return None
        r2 = op_map[op2](r1, c)
        if r2 is None: return None
        r3 = op_map[op3](r2, d)
        if r3 is None: return None
        return r3
    except Exception:
        return None

# ToT's "search": try every combination
# CoT "linearly" walks one path; ToT "branches" and tries multiple paths
target = 24
given = [3, 8, 8, 3]
ops = ['+', '-', '×', '÷']

print(f"=== Tree of Thoughts solves the 24-game ===")
print(f"Numbers: {given}")
print(f"Target: {target}\n")

# ToT: enumerate all permutations × all operation combinations
found = []
solutions_checked = 0
for perm in set(permutations(given)):
    for ops_seq in product(ops, repeat=3):
        solutions_checked += 1
        result = solve_24_to(perm, ops_seq)
        if result is not None and abs(result - target) < 1e-6:
            found.append((perm, ops_seq))

print(f"Search space: {solutions_checked} candidate paths")
print(f"Solutions found: {len(found)}\n")

for i, (perm, ops_seq) in enumerate(found[:3], 1):
    a, b, c, d = perm
    op1, op2, op3 = ops_seq
    print(f"  Solution #{i}: (({a} {op1} {b}) {op2} {c}) {op3} {d} = 24")

print(f"\nKey observation:")
print(f"  CoT is one linear path; if it goes wrong, everything is wrong")
print(f"  ToT models reasoning as a search tree; it can backtrack and parallelize")
print(f"  The cost is exponential growth in compute — only worth it for hard problems")
print(f"  LLM version of ToT: at each step, have the LLM generate k candidates, scored by an evaluator")


### 11.3 Verifier-guided Search: PRM Guidance

Tree of Thoughts needs to evaluate intermediate states. How to evaluate? Two kinds of verifiers:

- **ORM (Outcome Reward Model)**: only looks at whether the final answer is correct
- **PRM (Process Reward Model)**: scores every step

PRM is finer-grained than ORM — it can locate "which step went wrong." OpenAI's PRM800K dataset annotated the correctness of every step in 800,000 math problems; the PRM trained on it significantly outperforms ORM.

The verifier + search pipeline:

1. Sample a reasoning chain
2. The PRM scores every step
3. Low-scoring steps trigger resampling or backtracking
4. Continue until done

This is one of the core ideas behind the OpenAI o1 series, and also the verifier introduced in the later RL stage of DeepSeek-R1.


In [ ]:
# === PRM scoring demo: score every step of a reasoning chain ===
# Simulate: for a math problem's reasoning, the PRM scores every step and identifies which step went wrong

problem = "Xiaoming has 10 yuan. He buys 3 apples at 2 yuan each. How much money is left?"

# A reasoning chain (where Step 3 is computed wrong)
reasoning_chain = [
    {"step": 1, "text": "Money spent on apples = 3 × 2 = 6 yuan", "correct": True},
    {"step": 2, "text": "Originally had 10 yuan", "correct": True},
    {"step": 3, "text": "Remaining = 10 + 6 = 16 yuan", "correct": False},  # ← added wrong here
    {"step": 4, "text": "Answer: 16 yuan left", "correct": False},  # propagated error
]

# Simulate a PRM (Process Reward Model) scoring
def mock_prm_score(step_text, is_correct):
    """Simulate a PRM: a real model would learn many features — formula correctness, units, logical jumps, etc."""
    base = 0.5
    if is_correct:
        base += 0.4
    else:
        base -= 0.3
    # Extra signal: presence of operator symbols like '×' '+'
    if any(op in step_text for op in ['×', '+', '-', '÷', '=']):
        base += 0.05
    return round(base, 3)

# ORM (only looks at whether the final answer is correct)
def mock_orm_score(chain):
    final_correct = chain[-1]["correct"]
    return 0.9 if final_correct else 0.1

# Run PRM: score step by step
print(f"=== PRM step-by-step scoring ===")
print(f"Problem: {problem}\n")

step_scores = []
for s in reasoning_chain:
    score = mock_prm_score(s["text"], s["correct"])
    step_scores.append(score)
    status = "✓" if s["correct"] else "✗"
    print(f"  Step {s['step']} {status}  score {score:>5}  | {s['text']}")

# Run ORM: overall scoring
orm_score = mock_orm_score(reasoning_chain)

print(f"\n{'=' * 60}")
print(f"PRM final score (take min): {min(step_scores):.3f}")
print(f"PRM average score:          {sum(step_scores)/len(step_scores):.3f}")
print(f"ORM final score (by result): {orm_score:.3f}")
print(f"\nKey observation:")
print(f"  PRM can pinpoint Step 3 as the error (score {step_scores[2]})")
print(f"  ORM can only say 'the final answer is wrong' but not which step")
print(f"  → PRM-guided search can precisely backtrack to the bad step and resample")
print(f"  → DeepSeek-R1 introduces PRM in the later RL stage, significantly boosting math reasoning")


### 11.4 Self-Refine / Reflexion: Let the Model Fix Itself

The previous three methods all rely on external signals (RM, verifier). **Self-Refine** and **Reflexion** are different — they let the model evaluate and fix itself.

**Self-Refine** (Madaan et al. 2023) three-step loop:

1. **Generate**: the model first gives an answer
2. **Feedback**: have the same model evaluate where this answer is bad
3. **Refine**: have the model improve based on the feedback

**Reflexion** (Shinn et al. 2023) adds a "reflection memory" step:

1. The Actor generates an answer
2. The Evaluator scores it
3. The Self-Reflector generates a textual "reflection" ("the mistake I made last time was X")
4. Store the reflection in memory; use it as context the next time it generates
5. Retry on failure, each round carrying the reflections from previous rounds

The benefit of these two methods: **no need to train any extra model**. The downside: they depend on the model's self-evaluation ability; weak models self-evaluate poorly.


In [ ]:
# === Reflexion minimal demo: three rounds of reflection + retry ===
# Simulate: have the model write a piece of code, reflect after errors, then retry

import random
random.seed(42)

task = "Write a function: given a list, return the second-largest element"

# Simulate 3 rounds of Reflexion
rounds = [
    {
        "round": 1,
        "code": "def second_largest(lst):\n    return sorted(lst)[-2]",
        "feedback": "Missed edge cases: list length < 2 will raise an error; behavior with duplicates may be unexpected",
        "reflection": "Next time first check input length, and consider deduplication",
    },
    {
        "round": 2,
        "code": "def second_largest(lst):\n    if len(lst) < 2: return None\n    return sorted(lst)[-2]",
        "feedback": "Duplicates still a problem: [5,5,4] should return 4, but sorted[-2] returns 5",
        "reflection": "Deduplicate! Use set to remove duplicates before sorting",
    },
    {
        "round": 3,
        "code": "def second_largest(lst):\n    if len(lst) < 2: return None\n    uniq = sorted(set(lst))\n    if len(uniq) < 2: return None\n    return uniq[-2]",
        "feedback": "✓ Passes all tests",
        "reflection": "Success",
    },
]

print(f"=== Reflexion three-round iteration ===")
print(f"Task: {task}\n")

for r in rounds:
    print(f"--- Round {r['round']} ---")
    print(f"  Code:\n{r['code']}")
    print(f"  Feedback: {r['feedback']}")
    print(f"  Reflection: {r['reflection']}")
    print()

print(f"{'=' * 60}")
print(f"Key observation:")
print(f"  Round 1: ignored edge cases, [1] crashes")
print(f"  Round 2: edge cases fixed, but duplicates not handled")
print(f"  Round 3: dedup + edge cases, fully correct")
print(f"  → The model converges step by step via self-eval + reflection, with no external verifier")
print(f"  → Industrial practice: especially effective for code tasks (you can actually run tests)")
print(f"  → Open-ended tasks (copywriting) have limited effect: self-eval bias is large")


### 11.5 Four Methods vs Thinking Models

Putting the four inference-time methods covered in this section together with thinking models for comparison:

| Method | Changes model weights? | Extra compute factor | Key dependency | Suitable scenario |
|:---|:---|:---|:---|:---|
| Thinking model | Yes (RL training) | 1× (internal thinking) | High training cost | General reasoning |
| Self-Consistency | No | N× | Diverse sampling | Discrete, votable answers |
| Best-of-N + RM | No | N× | A good RM | Open-ended generation |
| Tree of Thoughts | No | k^d × | Node evaluator | Multi-step decisions |
| PRM-guided | No | N× + PRM | A high-quality PRM | Math, code |
| Self-Refine | No | 1 + K× | Model self-eval ability | Code, testable tasks |

Practical recommendations:

- **Tight budget, do not change the model**: Self-Consistency or Self-Refine (zero extra training)
- **Have an RM, pursue quality**: Best-of-N
- **Math/code scenarios**: PRM-guided, the R1 route
- **General reasoning, long-term investment**: train a thinking model

In real industrial deployment, multiple methods are often combined: first train a thinking model, then stack verifier-guided search at inference time — o1 / o3 are exactly this combination.


## Summary

1. ✅ **CoT** = have the model write out its reasoning process, using generated intermediate results to assist subsequent reasoning
2. ✅ **Thinking model** = engineered packaging of CoT; the thinking process may be separated by special tokens or API blocks
3. ✅ **Training pipeline**: Cold-start SFT → RL reasoning training → Rejection sampling → Full-scenario RL
4. ✅ **RL is the key**: do not prescribe "how to think" to the model, only reward "thought correctly"
5. ✅ **Reflection is emergent**: during RL the model may learn to check and correct itself on its own
6. ✅ **Each platform enables thinking differently**: OpenAI, DeepSeek, Qwen3, Claude all require checking the official docs for the corresponding model version
7. ✅ **Qwen3** is one of the few mainstream open-source models that explicitly supports runtime thinking/non-thinking switching
8. ✅ **Training yourself**: the teaching-version pipeline can run the concept through; the cost of a real high-quality thinking model cannot be summarized with a fixed price
9. ✅ **Easy route**: directly use DeepSeek-R1 distill models or the Qwen3 thinking model, ready out of the box

10. ✅ The five methods of test-time compute scaling: Self-Consistency / Best-of-N + RM / Tree of Thoughts / PRM-guided / Self-Refine
11. ✅ Self-Consistency: majority voting; effective when answers are discrete and comparable
12. ✅ Best-of-N: RM scores and picks the best; effective for continuous open-ended answers; the ceiling is set by the RM
13. ✅ Tree of Thoughts: models reasoning as a search tree, can backtrack; the cost is k^d × compute
14. ✅ PRM vs ORM: PRM scores step by step and can locate errors; it is a key component of o1 / R1
15. ✅ Self-Refine / Reflexion: model self-eval + reflection + retry; no need to train an external model
16. ✅ Practice: thinking model + verifier-guided search are often combined (o1 / o3)

**One-sentence summary**: a regular model = gives the answer directly; a thinking model = drafts first, then answers.
The drafting behavior can come from the prompt, from SFT data, or be reinforced out of RL on verifiable tasks.
Now you know how to check the API switches, and also why training a thinking model is not just about adding `<think>` tags.


## Exercises

> You can ask AI to help explain the approach, but it is not recommended to let AI "do this problem for you."


**Exercise 1: The Relationship Between CoT Reasoning Steps and Accuracy**

Chain-of-Thought improves reasoning accuracy by having the model "think step by step." But CoT is not "the longer the better."

| Reasoning steps | Accuracy |
|:---|:---|
| 0 (answer directly) | 45% |
| 1-3 steps | 62% |
| 4-8 steps | 78% |
| 9-15 steps | 80% |
| 16+ steps | 77% |

Analyze: why does accuracy decrease after the number of reasoning steps exceeds a certain range?

Hint: errors accumulate in long reasoning chains — if some intermediate step is computed wrong, all subsequent steps are led astray.


In [ ]:
# Exercise 1: The Relationship Between CoT Reasoning Steps and Accuracy
answer = "fill your answer here"

# A) The more steps the better; 77% is just experimental noise
# B) Errors accumulate in long reasoning chains, and long sequences disperse attention, which is bad for accurate reasoning
# C) The model has limited ability; reasoning beyond 15 steps exceeds the model's capacity
# D) The dataset is too simple and does not need that many steps

assert not answer.startswith("fill your answer here"), "Please fill in your answer first"
assert answer in "ABCD", "Please fill in one of A/B/C/D"

if answer == "B":
    print("✅ Exercise 1 passed:")
    print("   The core of CoT improving accuracy is giving the model room to show intermediate reasoning.")
    print("   But when the steps are too long:")
    print("   1. Errors in intermediate steps accumulate along the reasoning chain")
    print("   2. Attention efficiency drops in very long contexts")
    print("   3. The more tokens generated, the higher the inference cost")
    print("   So CoT needs to balance 'sufficient reasoning' against 'the risk of chained errors'.")
else:
    print(f"You chose {answer}.")
    print("Hint: think about the two-sided nature of long reasoning chains — they give more reasoning room but also bring error accumulation.")


**Exercise 2: Ordering the Training Stages of a Thinking Model**

The training of a thinking model (such as DeepSeek-R1) is usually divided into four stages:
1. RL reasoning training
2. Cold-start SFT
3. Full-scenario RL
4. Rejection sampling + SFT

Please arrange these four stages in the correct training order.

Hint: first SFT to lay the foundation (learn the basic format), then RL to boost reasoning, then rejection sampling to distill, and finally full-scenario generalization.


In [ ]:
# Exercise 2: Ordering the Training Stages of a Thinking Model

# Fill in the list of stage numbers, e.g. [2, 1, 4, 3]
# 1 = RL reasoning training
# 2 = Cold-start SFT
# 3 = Full-scenario RL
# 4 = Rejection sampling + SFT

order = None  # Fill in the list of the correct order here

assert order is not None, "Please fill in the correct order"
assert len(order) == 4, "Need to arrange 4 stages"
assert set(order) == {1, 2, 3, 4}, "Must include all four stages 1, 2, 3, 4"

correct_order = [2, 1, 4, 3]
if order == correct_order:
    print("✅ Exercise 2 passed:")
    print("   1. Cold-start SFT → learn the basic CoT format")
    print("   2. RL reasoning training → reinforce reasoning ability on verifiable tasks")
    print("   3. Rejection sampling + SFT → filter high-quality reasoning data from the RL model")
    print("   4. Full-scenario RL → generalize to conversation, writing, and other general scenarios")
else:
    print(f"Your order: {order}, correct order: {correct_order}")
    print("Hint: SFT is always done first (to lay the foundation); full-scenario RL is last (to generalize).")


**Exercise 3: Reward Function Design**

The reward scheme when training a thinking model:

| Case | Reward |
|:---|:---|
| Correct answer | +1.0 |
| Wrong answer | -1.0 |
| Correct format | +0.1 |
| Language inconsistent | -0.1 |

Compute the total reward for the following 5 samples:
1. Correct answer, correct format, consistent language
2. Wrong answer, correct format, inconsistent language
3. Correct answer, wrong format, consistent language
4. Wrong answer, wrong format, inconsistent language
5. Correct answer, correct format, inconsistent language

Hint: total reward = sum of all items. E.g. case 1 = +1.0 + 0.1 = +1.1.


In [ ]:
# Exercise 3: Reward Function Design
samples = [
    {"answer": "correct", "format": True,  "lang": "consistent"},
    {"answer": "wrong",   "format": True,  "lang": "inconsistent"},
    {"answer": "correct", "format": False, "lang": "consistent"},
    {"answer": "wrong",   "format": False, "lang": "inconsistent"},
    {"answer": "correct", "format": True,  "lang": "inconsistent"},
]

def compute_reward(s):
    r = 1.0 if s["answer"] == "correct" else -1.0
    if s["format"]: r += 0.1
    if s["lang"] == "inconsistent": r += -0.1
    return r

# TODO: compute the total reward for each sample
rewards = None  # a list of 5 reward values

assert rewards is not None, "Please compute the rewards first"
assert len(rewards) == 5, "Need 5 reward values"

expected = [compute_reward(s) for s in samples]
for i, (r, e) in enumerate(zip(rewards, expected)):
    assert abs(r - e) < 0.01, f"Sample {i+1} reward should be {e:.1f}, you got {r:.1f}"

print("✅ Exercise 3 passed:")
for i, r in enumerate(rewards):
    print(f"   Sample {i+1}: reward = {r:+.1f}")
print()
print("   The reward function guides the model: prioritize correct answers (±1.0), while also watching format and language.")
